# AGR-CLIP v2 — Gated Reconstruction Pooling + Differential LR

1 epoch, batch 16, CLIP LR 2e-7, reconstruction LR 2e-6. The reconstruction decoder is unchanged; token pooling is learned. NO random masking, NO auxiliary token loss. Requires fresh original PR-CLIP.zip, not STPR zip.


In [1]:
from google.colab import drive
drive.mount('/content/drive')

# upload PR-CLIP.zip lên Drive trước, sửa đường dẫn cho đúng
!rm -rf /content/PR-CLIP
!unzip -q -o /content/drive/MyDrive/PR-CLIP.zip -d /content
%cd /content/PR-CLIP

Mounted at /content/drive
/content/PR-CLIP


In [2]:
!pip install -q open_clip_torch==2.20.0 clip-benchmark==1.6.1 transformers==4.45.2 adapters timm sentence-transformers wandb pyyaml gdown
!pip install -q "protobuf>=6.31.1,<7"

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-cloud-datastore 2.27.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 3.20.3 which is incompatible.
proto-plus 1.29.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 3.20.3 which is incompatible.
google-api-core 2.30.3 requires protobuf<8.0.0,>=4.25.8, but you have protobuf 3.20.3 which is incompatible.
google-cloud-functions 1.25.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 3.20.3 which is incompatible.
google-cloud-firestore 2.34.1 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 3.20.3 which is incompatible.
google-cloud-storage-control 1.17.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 3.20.3 which is incompatible.
google-cloud-bigquery-connection 1.23.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 3.20.3 which is incompatible.
google-cloud-mon

In [3]:
!gdown --fuzzy "https://drive.google.com/file/d/15htMe2iE4kVujjWURxfCxZXk8vAAetoT/view" -O images.zip
!rm -rf images && mkdir images
!tar -xzf images.zip -C images
!find images -type f -exec mv -t images {} + 2>/dev/null
!find images -mindepth 1 -type d -empty -delete
!rm images.zip
!mkdir -p cache/weights

Downloading...
From (original): https://drive.google.com/uc?id=15htMe2iE4kVujjWURxfCxZXk8vAAetoT
From (redirected): https://drive.google.com/uc?id=15htMe2iE4kVujjWURxfCxZXk8vAAetoT&confirm=t&uuid=91605fad-6283-4053-ac4f-c756f465d69e
To: /content/PR-CLIP/images.zip
100% 1.46G/1.46G [00:31<00:00, 45.7MB/s]


In [4]:
!mkdir -p /content/PR-CLIP/cache/weights
!wget -O /content/PR-CLIP/cache/weights/RET-2.pt \
  https://huggingface.co/Zilun/GeoRSCLIP/resolve/main/ckpt/RS5M_ViT-B-32_RET-2.pt
!ls -la /content/PR-CLIP/cache/weights/RET-2.pt   # khoảng 605MB

--2026-11-03 14:24:28--  https://huggingface.co/Zilun/GeoRSCLIP/resolve/main/ckpt/RS5M_ViT-B-32_RET-2.pt
Resolving huggingface.co (huggingface.co)... 18.239.50.80, 18.239.50.16, 18.239.50.49, ...
Connecting to huggingface.co (huggingface.co)|18.239.50.80|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://us.gcp.cdn.hf.co/xet-bridge-us/655fb2aadc7511e22748819d/48ec26e685f2b7622075158c0649039d64cdcbdbe150faedb6e79ea1d13d0e78?X-Xet-Cas-Uid=public&xip=gCWL4-6pDZI&user_id=public&response-content-disposition=inline%3B+filename*%3DUTF-8%27%27RS5M_ViT-B-32_RET-2.pt%3B+filename%3D%22RS5M_ViT-B-32_RET-2.pt%22%3B&Expires=1793719468&Policy=eyJTdGF0ZW1lbnQiOlt7IlJlc291cmNlIjoiaHR0cHM6Ly91cy5nY3AuY2RuLmhmLmNvL3hldC1icmlkZ2UtdXMvNjU1ZmIyYWFkYzc1MTFlMjI3NDg4MTlkLzQ4ZWMyNmU2ODVmMmI3NjIyMDc1MTU4YzA2NDkwMzlkNjRjZGNiZGJlMTUwZmFlZGI2ZTc5ZWExZDEzZDBlNzhcXD9YLVhldC1DYXMtVWlkPXB1YmxpYyZ4aXA9Z0NXTDQtNnBEWkkmdXNlcl9pZD1wdWJsaWMmcmVzcG9uc2UtY29udGVudC1kaXNwb3NpdGlvbj0qIiwiQ29u

In [5]:
import pandas as pd, os
for f in ['dataset/train/ret2.csv','dataset/test/rsicd_test.csv','dataset/test/rsitmd_test.csv']:
    names = pd.read_csv(f)['filename'].unique()
    missing = [n for n in names if not os.path.exists(os.path.join('images', n))]
    print(f, '| tổng:', len(names), '| thiếu:', len(missing))

dataset/train/ret2.csv | tổng: 13092 | thiếu: 0
dataset/test/rsicd_test.csv | tổng: 1093 | thiếu: 0
dataset/test/rsitmd_test.csv | tổng: 452 | thiếu: 0


In [6]:
%cd /content/PR-CLIP
!sed -i 's/from timm.optim.nadam import Nadam/from timm.optim.nadam import NAdamLegacy as Nadam/' training/optimization.py
!sed -i 's/from timm.optim.radam import RAdam$/from timm.optim.radam import RAdamLegacy as RAdam/' training/optimization.py
!sed -n 5,12p training/optimization.py
!nvidia-smi | head -12
!sed -i 's/np\.string_/np.bytes_/g' data/train_data.py utils/tsne_checkpoints.py
!sed -i 's/astype(np\.float)/astype(float)/g' evaluation/linear_eval.py
!grep -n "np.bytes_" data/train_data.py

/content/PR-CLIP
from timm.optim.adafactor import Adafactor
from timm.optim.adahessian import Adahessian
from timm.optim.adamp import AdamP
from timm.optim.lookahead import Lookahead
from timm.optim.nadam import NAdamLegacy as Nadam
from timm.optim.radam import RAdamLegacy as RAdam
from timm.optim.rmsprop_tf import RMSpropTF
from timm.optim.sgdp import SGDP
Tue Nov  3 14:24:35 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+=============

In [7]:
!pip install -q cn_clip --no-deps
!nvidia-smi
!pip install -q faiss-cpu click

Tue Nov  3 14:24:39 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   46C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Repository compatibility fixes from your original notebook

In [8]:
%cd /content/PR-CLIP
import re

def read(p): return open(p, encoding='utf-8').read().replace('\r\n', '\n')
def write(p, s): open(p, 'w', encoding='utf-8').write(s)
def sub(path, pairs):
    s = read(path)
    for old, new in pairs:
        s = s.replace(old, new)
    write(path, s)


# 2. timm / numpy bản mới đổi tên + tắt log dài
sub('training/optimization.py', [
    ('from timm.optim.nadam import Nadam\n', 'from timm.optim.nadam import NAdamLegacy as Nadam\n'),
    ('from timm.optim.radam import RAdam\n', 'from timm.optim.radam import RAdamLegacy as RAdam\n'),
    ('logging.info("Param groups = %s" % json.dumps(parameter_group_names, indent=2))', 'pass'),
])
sub('data/train_data.py', [('np.string_', 'np.bytes_')])
sub('utils/tsne_checkpoints.py', [('np.string_', 'np.bytes_')])
sub('evaluation/linear_eval.py', [('astype(np.float)', 'astype(float)')])
sub('training/main.py', [
    ("logging.info(f'\\t{n}\\t{list(p.size())}')", 'pass'),
    ("logging.info('Model structure\\n' +str(model))", 'pass'),
    ('logging.info(f"  {name}: {val}")', 'pass'),
])

# 3. Thay return_hidden của open_clip bản sửa
helper = '''    def _vit_hidden(self, images):
        v = self.image_backbone
        x = v.conv1(images.type(v.conv1.weight.dtype))
        x = x.reshape(x.shape[0], x.shape[1], -1).permute(0, 2, 1)
        cls = v.class_embedding.to(x.dtype) + torch.zeros(x.shape[0], 1, x.shape[-1], dtype=x.dtype, device=x.device)
        x = torch.cat([cls, x], dim=1)
        x = x + v.positional_embedding.to(x.dtype)
        x = v.patch_dropout(x)
        x = v.ln_pre(x)
        x = x.permute(1, 0, 2)
        x = v.transformer(x)
        x = x.permute(1, 0, 2)
        return v.ln_post(x)

'''
s = read('model/model.py')
if '_vit_hidden' not in s:
    s = s.replace("self.image_backbone(images, return_hidden=True)", "self._vit_hidden(images)")
    marker = "    def forward(self, images, texts, text_only=False):"
    s = s.replace(marker, helper + marker)
    write('model/model.py', s)

# 4. In log theo % epoch (10 -> 10%,20%,...; 50 -> 50%,100%)
LOG_PCT = 10
s = read('training/train.py')
s = s.replace("(i % 10 == 0 or batch_count == num_batches_per_epoch)",
              "(batch_count in {round(num_batches_per_epoch * k * LOG_PCT / 100) for k in range(1, 100 // LOG_PCT + 1)})")
if 'LOG_PCT = ' not in s:
    anchor = "import torch.nn.functional as F\n"
    s = s.replace(anchor, anchor + f"\nLOG_PCT = {LOG_PCT}\n", 1)
else:
    s = re.sub(r"LOG_PCT = \d+", f"LOG_PCT = {LOG_PCT}", s, count=1)
write('training/train.py', s)


# Kiểm tra: phải ra 2, 1, 2, 2, True
!grep -c "_vit_hidden" model/model.py
!grep -c "NAdamLegacy" training/optimization.py
!grep -c "np.bytes_" data/train_data.py
!grep -c "LOG_PCT" training/train.py
print('[DBG-fp32]' in read('training/train.py'))


%cd /content/PR-CLIP
import re, py_compile
p = 'training/main.py'
s = open(p, encoding='utf-8').read().replace('\r\n', '\n')

if '[LITE-CKPT]' in s:
    print('Đã vá từ trước, không làm gì thêm.')
else:
    def sub_regex(pattern, repl, text, name):
        n = len(re.findall(pattern, text))
        assert n == 1, f'[{name}] khớp {n} chỗ (cần đúng 1)'
        return re.sub(pattern, repl, text, count=1)

    def sub_literal(old, new, text, name):
        n = text.count(old)
        assert n == 1, f'[{name}] khớp {n} chỗ (cần đúng 1)'
        return text.replace(old, new)

    # 1. Không lưu epoch_N.pt (chỉ còn best.pt + epoch_latest.pt)
    s = sub_literal('if completed_epoch == args.epochs or (', 'if False or (', s, 'epoch_N')

    # 2. epoch_latest.pt: ĐẦY ĐỦ (fp32 + optimizer + scaler), ghi qua file .tmp rồi đổi tên
    pat = r'( +)if args\.save_most_recent:\n\s+torch\.save\(\n\s+checkpoint_dict,\n\s+os\.path\.join\(args\.checkpoint_path, f"epoch_latest\.pt"\),\n\s+\)[ ]*\n'
    def rep_latest(m):
        i = m.group(1)
        return (f'{i}if args.save_most_recent:  # [LITE-CKPT]\n'
                f'{i}    _tmp = os.path.join(args.checkpoint_path, "epoch_latest.tmp")\n'
                f'{i}    torch.save(checkpoint_dict, _tmp)\n'
                f'{i}    os.replace(_tmp, os.path.join(args.checkpoint_path, "epoch_latest.pt"))\n')
    s = sub_regex(pat, rep_latest, s, 'epoch_latest')

    # 3. best.pt: NHẸ (fp16, không optimizer)
    pat = r'torch\.save\(\n(\s+)checkpoint_dict,\n(\s+)os\.path\.join\(args\.checkpoint_path, f"best\.pt"\),'
    def rep_best(m):
        return ('torch.save(\n' + m.group(1) +
                '{**checkpoint_dict, "optimizer": None, "state_dict": {k: (v.half() if v.is_floating_point() else v) for k, v in checkpoint_dict["state_dict"].items()}},\n' +
                m.group(2) + 'os.path.join(args.checkpoint_path, f"best.pt"),')
    s = sub_regex(pat, rep_best, s, 'best')

    # 4. Resume được cả khi checkpoint không có optimizer (vd best.pt nhẹ)
    s = sub_literal('                    if optimizer is not None:\n                        optimizer.load_state_dict(checkpoint["optimizer"])',
                    '                    if optimizer is not None and checkpoint.get("optimizer") is not None:\n                        optimizer.load_state_dict(checkpoint["optimizer"])',
                    s, 'resume')

    # 5. Train xong đủ epoch -> tự xóa epoch_latest (giữ best.pt)
    old = '    if args.wandb and is_master(args):\n        wandb.finish()\n'
    new = ('    if is_master(args) and args.save_logs:  # [AUTO-DEL]\n'
           '        for _f in ("epoch_latest.pt", "epoch_latest.tmp"):\n'
           '            _p = os.path.join(args.checkpoint_path, _f)\n'
           '            if os.path.exists(_p):\n'
           '                os.remove(_p)\n'
           '                logging.info(f"Train xong, da xoa {_p}")\n') + old
    s = sub_literal(old, new, s, 'auto_delete')

    open(p, 'w', encoding='utf-8').write(s)
    py_compile.compile(p, doraise=True)   # báo lỗi cú pháp ngay nếu vá sai
    print('Đã vá xong:', '[LITE-CKPT]' in s, '[AUTO-DEL]' in s)
    print('best.pt nhẹ (fp16, không optimizer) | epoch_latest.pt đầy đủ | tự xóa epoch_latest khi xong')

# ---- Vá thêm: resume trên PyTorch >= 2.6 + dừng ngay nếu thiếu checkpoint ----
import py_compile
p = 'training/main.py'
s = open(p, encoding='utf-8').read().replace('\r\n', '\n')

def _sub_once(old, new, text, name):
    n = text.count(old)
    assert n == 1, f'[{name}] khớp {n} chỗ (cần đúng 1)'
    return text.replace(old, new)

if 'weights_only=False' not in s:
    s = _sub_once('torch.load(args.resume, map_location=device)',
                  'torch.load(args.resume, map_location=device, weights_only=False)',
                  s, 'weights_only')
if 'Khong tim thay checkpoint' not in s:
    s = _sub_once('            logging.info("=> no checkpoint found at \'{}\'".format(args.resume))',
                  '            raise FileNotFoundError(f"Khong tim thay checkpoint: {args.resume}")',
                  s, 'resume_missing')
open(p, 'w', encoding='utf-8').write(s)
py_compile.compile(p, doraise=True)
print('Vá resume OK:', 'weights_only=False' in s, 'Khong tim thay checkpoint' in s)

!grep -n "load-clip-weights\"" training/params.py   # phải thấy default=True



/content/PR-CLIP
2
1
2
2
False
/content/PR-CLIP
Đã vá xong: True True
best.pt nhẹ (fp16, không optimizer) | epoch_latest.pt đầy đủ | tự xóa epoch_latest khi xong
Vá resume OK: True True
450:        "--load-clip-weights", default=True, action="store_true",


## AGR-v2 reconstruction modification + corrected AMP accumulation

In [9]:
%cd /content/PR-CLIP
from pathlib import Path
import py_compile
p=Path('model/model.py');s=p.read_text()
old='                text_features = open_clip_forward(texts)'
new='''                text_features = open_clip_forward(texts)
                self._agr_text_mask = texts.ne(0).long()
                if self.prompt is not None:
                    extra = torch.ones(texts.size(0), self.n_prompt, device=texts.device, dtype=torch.long)
                    self._agr_text_mask = torch.cat([self._agr_text_mask[:, :1], extra, self._agr_text_mask[:, 1:]], dim=1)
'''
assert s.count(old)==1;s=s.replace(old,new,1)
old='        text_masks = torch.ones(text_embed.size()[:-1],dtype=torch.long).to(self.device)'
new='''        text_masks = self._agr_text_mask.to(text_embed.device)
        assert text_masks.shape == text_embed.shape[:2], (text_masks.shape, text_embed.shape)'''
assert s.count(old)==1;s=s.replace(old,new,1)
cls='''class AGRPooling(nn.Module):
    """Mask-aware attention pooling with learnable gated mean residual."""
    def __init__(self, dim=512):
        super().__init__()
        self.score = nn.Linear(dim, 1)
        nn.init.zeros_(self.score.weight)
        nn.init.zeros_(self.score.bias)
        self.gate_logit = nn.Parameter(torch.tensor(-2.0))

    def forward(self, hidden, mask):
        valid = mask.bool()
        scores = self.score(hidden.float()).squeeze(-1)
        scores = scores.masked_fill(~valid, torch.finfo(scores.dtype).min)
        weight = torch.softmax(scores, dim=1).to(hidden.dtype)
        attention_pool = (weight.unsqueeze(-1) * hidden).sum(dim=1)
        validf = valid.unsqueeze(-1).to(hidden.dtype)
        mean_pool = (hidden * validf).sum(1) / validf.sum(1).clamp_min(1)
        gate = torch.sigmoid(self.gate_logit).to(hidden.dtype)
        return (1 - gate)*mean_pool + gate*attention_pool

'''
anchor='class ItraModel(nn.Module):';assert s.count(anchor)==1;s=s.replace(anchor,cls+anchor,1)
anchor='        self.fusion_model = FusionModel(pretrained_path=None)';assert s.count(anchor)==1;s=s.replace(anchor,anchor+'\n        self.agr_image_pool = AGRPooling(dim_str)\n        self.agr_text_pool = AGRPooling(dim_str)',1)
old='''        image_output = self.fusion_model(image_res_embed, image_masks, text_embed, text_masks)
        text_output = self.fusion_model(text_res_embed, text_masks, image_embed, image_masks)

        image_rec_features = F.normalize(torch.mean(image_output, dim=1), dim=-1)
        text_rec_features = F.normalize(torch.mean(text_output, dim=1), dim=-1)'''
new='''        # AGR-v2: keep all reconstruction queries intact, learn token importance.
        image_output = self.fusion_model(image_res_embed, image_masks, text_embed, text_masks)
        text_output = self.fusion_model(text_res_embed, text_masks, image_embed, image_masks)

        image_rec_features = F.normalize(self.agr_image_pool(image_output, image_masks), dim=-1)
        text_rec_features = F.normalize(self.agr_text_pool(text_output, text_masks), dim=-1)'''
assert s.count(old)==1;s=s.replace(old,new,1)
assert 'self.total_loss = rec_loss # + rec_loss2' in s
p.write_text(s)
p=Path('training/optimization.py');s=p.read_text()
needle='        if group_name not in parameter_group_names:'
replacement='''        # Different LR: reconstruction 10x, CLIP backbone 1x.
        if name.startswith(('fusion_model.', 'it_encoder.', 'proj_image.', 'proj_text.', 'fc_image.', 'fc_text.', 'agr_image_pool.', 'agr_text_pool.')):
            group_name = 'reconstruction_' + group_name
        if group_name not in parameter_group_names:'''
assert s.count(needle)==1;s=s.replace(needle,replacement,1)
needle='            parameter_group_names[group_name] = {'
assert s.count(needle)==1;s=s.replace(needle,"            if group_name.startswith('reconstruction_'):\n                scale *= 10.0\n"+needle,1)
p.write_text(s)
p=Path('training/train.py');s=p.read_text()
needle='    step_lou = 4\n';assert s.count(needle)==1;s=s.replace(needle,needle+'    optimizer.zero_grad(set_to_none=True)  # AGR-AMP-FIX\n    norm = torch.tensor(0.0)\n',1)
start='            if scaler is not None:\n                scaler.scale(total_loss).backward()'
end='            batch_time_m.update(time.time() - end)'
assert s.count(start)==1 and s.count(end)==1
a=s.index(start);b=s.index(end,a)
replacement='''            if scaler is not None:
                scaler.scale(total_loss).backward()
            else:
                total_loss.backward()
            _step_now = ((i + 1) % step_lou == 0 or i == train_loader_size - 1)
            if _step_now:
                if scaler is not None:
                    scaler.unscale_(optimizer)
                norm = nn.utils.clip_grad_norm_(model.parameters(), max_norm=args.max_grad_norm)
                if scaler is not None:
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    optimizer.step()
                optimizer.zero_grad(set_to_none=True)

'''
s=s[:a]+replacement+s[b:]
s=s.replace('f"LR: {optimizer.param_groups[0][\'lr\']:3f} "','f"LR: {optimizer.param_groups[0][\'lr\']:.3e} "')
p.write_text(s)
for f in ('model/model.py','training/optimization.py','training/train.py'):
 py_compile.compile(f,doraise=True)
print('AGR patch: compiled 3 files')


/content/PR-CLIP
AGR patch: compiled 3 files


In [10]:
from pathlib import Path
for name in ['model/model.py','training/train.py','training/optimization.py']:
    import py_compile
    py_compile.compile(name, doraise=True)
    print('CHECK OK:',name)
assert 'class AGRPooling' in Path('model/model.py').read_text()
assert 'scale *= 10.0' in Path('training/optimization.py').read_text()
print('No STPR random masking; one epoch configured')

CHECK OK: model/model.py
CHECK OK: training/train.py
CHECK OK: training/optimization.py
No STPR random masking; one epoch configured


## One epoch only — new experiment output directory

Compare results to RET-2 untrained (mR 38.88) and original PR-CLIP (mR 38.70). Improvement is a hypothesis, not guaranteed. This code corrects the AMP training path so old results are not perfectly controlled baselines.

In [11]:
%cd /content/PR-CLIP
!torchrun --nproc_per_node 1 -m training.main \
  --train-data ./dataset/train/ret2.csv --images-dir ./images \
  --retrieval-data ./dataset/test/rsicd_test.csv --retrieval-images-dir ./images \
  --csv-separator , --csv-img-key filename \
  --retrieval-csv-separator , --retrieval-csv-img-key filename \
  --retrieval-frequency 1 --save-frequency 0 \
  --epochs 1 --batch-size 16 --workers 2 --precision amp \
  --lr 2e-7 --warmup 100 --weight_decay 0.5 --max-grad-norm 50 \
  --image-model ViT-B-32 --image-model-builder openclip \
  --text-model ViT-B-32 --text-model-builder openclip \
  --pretrained-image-model --pretrained-text-model --loss InfoNCE \
  --seed 1 \
  --name rsicd_AGR_gatepool_dlr10_bs16_ep1_v2 --logs /content/drive/MyDrive/prclip_logs

/content/PR-CLIP
I0000 00:00:1793715896.722912    1915 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1793715899.047649    1915 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
/usr/local/lib/python3.13/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
/content/PR-CLIP/evaluation/SentEval/senteval/utils.py:62: SyntaxWarning: invalid escape sequence '\d'
  assert re.match("^[+-]?(\d+(\.\d*)?|\.\d+)$", split[1]) is not None
Namespace(config_yaml=None, text_model_builder='openclip', text_model='ViT-B-32', text_pooler='cls', text_mo